# 逐次予測

前処理済みのデータを用いてモデルを構築し、評価する

- 目的変数: `price_actual`
- モデル: LightGBM, RandomForest, SVR, NeuralNetwork のいずれか
- 評価指標: RMSE
- ハイパーパラメータチューニング: ベイズ最適化

## 1. ライブラリのインポートとデータ読み込み

In [1]:
# 自動ローディング
%load_ext autoreload
%autoreload 2

In [2]:
# アクティベート
!source ../../.venv/bin/activate

In [3]:
# LightGBM特有のエラー対策
#!brew install libomp
#!pip uninstall lightgbm
#!pip install lightgbm

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# データ読み込み用の関数をインポート
import sys, os
sys.path.append(os.pardir)  # 親ディレクトリのファイルをインポートするための設定
from src.data_loader import check_missing_values
from src.modeling import train_and_predict

# データディレクトリ
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
SUBMISSION = DATA_DIR / 'submission'

os.makedirs(SUBMISSION, exist_ok=True)

print(DATA_DIR)
# 前処理済みデータの読み込み
train = pd.read_csv(DATA_DIR / 'train_processed.csv')
test = pd.read_csv(DATA_DIR / 'test_processed.csv')

print('train shape:', train.shape)
print('test shape:', test.shape)

/Users/m0122wt/Desktop/02.プライベート/01.ノウハウ/07.データ分析/notebook/signate_smbc_202506/data
train shape: (26280, 94)
test shape: (8760, 93)


## 2. 特徴量・目的変数の設定

In [5]:
# 目的変数
target_col = 'price_actual'

# 説明変数（目的変数とtime列以外）
drop_cols = ['time', target_col] if target_col in train.columns else ['time']
feature_cols = [col for col in train.columns if col not in drop_cols]

X = train[feature_cols]
y = train[target_col] if target_col in train.columns else train.iloc[:, -1]  # 念のため

print('Features:', feature_cols)
print('Target:', target_col)
print('X shape:', X.shape)
print('y shape:', y.shape)

Features: ['generation_biomass', 'generation_fossil_brown_coal/lignite', 'generation_fossil_gas', 'generation_fossil_hard_coal', 'generation_fossil_oil', 'generation_hydro_pumped_storage_consumption', 'generation_hydro_run_of_river_and_poundage', 'generation_hydro_water_reservoir', 'generation_nuclear', 'generation_other', 'generation_other_renewable', 'generation_solar', 'generation_waste', 'generation_wind_onshore', 'total_load_actual', 'valencia_pressure', 'valencia_humidity', 'valencia_wind_speed', 'valencia_wind_deg', 'valencia_rain_1h', 'valencia_rain_3h', 'valencia_snow_3h', 'valencia_clouds_all', 'madrid_pressure', 'madrid_humidity', 'madrid_wind_speed', 'madrid_wind_deg', 'madrid_rain_1h', 'madrid_rain_3h', 'madrid_snow_3h', 'madrid_clouds_all', 'bilbao_pressure', 'bilbao_humidity', 'bilbao_wind_speed', 'bilbao_wind_deg', 'bilbao_rain_1h', 'bilbao_rain_3h', 'bilbao_snow_3h', 'bilbao_clouds_all', 'barcelona_pressure', 'barcelona_humidity', 'barcelona_wind_speed', 'barcelona_win

In [6]:
# 学習とテストデータでカラムの構成に違いがないか確認
diff_features = set(train.columns) - set(test.columns)
print("差分があるカラム:")
print(sorted(list(diff_features)))

# 欠損値の確認
check_missing_values(train)
check_missing_values(test)

差分があるカラム:
['price_actual']
欠損値があるカラム、欠損値の数、全レコードに対する割合:


,missing_count,missing_ratio


欠損値があるカラム、欠損値の数、全レコードに対する割合:


,missing_count,missing_ratio


## 3. 学習・検証

In [ ]:
%%time
filename = 'submission_nn'

# 通常の予測
model, predictions = train_and_predict(
    model_type='neural_network',
    train_df=train,
    test_df=test,
    target_col='price_actual',
    optimize=True,
    sequential=False,  # 通常予測
    output_path=SUBMISSION,
    filename=filename
)

137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


[I 2025-06-29 19:28:31,277] Trial 0 finished with value: 4.230525019088106 and parameters: {'layer1_units': 161, 'layer2_units': 97, 'layer3_units': 43, 'dropout_rate': 0.15593898501118525, 'learning_rate': 0.0031550822395662132, 'batch_size': 16}. Best is trial 0 with value: 4.230525019088106.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 787us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 651us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 707us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 692us/step


[I 2025-06-29 19:29:39,251] Trial 1 finished with value: 6.310324773774258 and parameters: {'layer1_units': 70, 'layer2_units': 116, 'layer3_units': 50, 'dropout_rate': 0.1488832508429328, 'learning_rate': 0.0004478445203864317, 'batch_size': 32}. Best is trial 0 with value: 4.230525019088106.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 650us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 807us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 726us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 668us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 816us/step


...

137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 637us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 646us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 809us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 674us/step


[I 2025-06-30 00:35:58,860] Trial 98 finished with value: 3.6971392596244996 and parameters: {'layer1_units': 88, 'layer2_units': 83, 'layer3_units': 51, 'dropout_rate': 0.1382797753393795, 'learning_rate': 0.008878697502776495, 'batch_size': 16}. Best is trial 62 with value: 3.5697693374659956.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 727us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 659us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 680us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 733us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 849us/step


[I 2025-06-30 00:37:54,237] Trial 99 finished with value: 3.9706310968425114 and parameters: {'layer1_units': 83, 'layer2_units': 81, 'layer3_units': 45, 'dropout_rate': 0.14701880532967096, 'learning_rate': 0.006667083946223166, 'batch_size': 16}. Best is trial 62 with value: 3.5697693374659956.


NEURAL_NETWORK：ベイズ最適化の結果
Best params: {'layer1_units': 121, 'layer2_units': 59, 'layer3_units': 54, 'dropout_rate': 0.13383472933292856, 'learning_rate': 0.00830555831035743, 'batch_size': 16, 'hidden_layers': [121, 59, 54], 'epochs': 100, 'patience': 10}
Best CV RMSE: 3.5697693374659956
Epoch 1/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 5s 558us/step - loss: 187.1805 - mae: 9.7255 - learning_rate: 0.0083
Epoch 2/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 621us/step - loss: 72.2817 - mae: 6.6152 - learning_rate: 0.0083
Epoch 3/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 567us/step - loss: 47.7895 - mae: 5.3363 - learning_rate: 0.0083
Epoch 4/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 651us/step - loss: 40.4110 - mae: 4.8910 - learning_rate: 0.0083
Epoch 5/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 551us/step - loss: 36.6129 - mae: 4.6500 - learning_rate: 0.0083
Epoch 6/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 543us/step - loss: 34.8053 - mae: 4.5237 - learning_rate: 0.0083
Epoch 7/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1

### 履歴
___
LIGHTGBM：ベイズ最適化の結果
Best params: {'num_leaves': 20, 'learning_rate': 0.07056499143559153, 'feature_fraction': 0.930666633751073, 'bagging_fraction': 0.8764523692423547, 'bagging_freq': 9, 'min_child_samples': 35, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1, 'random_state': 42}  
Best CV RMSE: 2.517122779389563  
Selected features: 73/92  
___
NEURAL_NETWORK：ベイズ最適化の結果
Best params: {'layer1_units': 48, 'layer2_units': 121, 'layer3_units': 8, 'dropout_rate': 0.16207371516723865, 'learning_rate': 0.005173979484789956, 'batch_size': 16}  
Best CV RMSE: 3.7179227519163134  
___
NEURAL_NETWORK：ベイズ最適化の結果
Best params: {'layer1_units': 121, 'layer2_units': 59, 'layer3_units': 54, 'dropout_rate': 0.13383472933292856, 'learning_rate': 0.00830555831035743, 'batch_size': 16, 'hidden_layers': [121, 59, 54], 'epochs': 100, 'patience': 10}  
Best CV RMSE: 3.5697693374659956  
___


In [ ]:
predictions

array([11.300329, 12.014666, 12.260893, ..., 20.831654, 20.7573  ,
       20.324287], dtype=float32)

In [ ]:
%%time

filename_sequential = 'submission_nn_sequential'

# 逐次予測（ラグ特徴量を考慮）
model_sequential, predictions_sequential = train_and_predict(
    model_type='neural_network',
    train_df=train,
    test_df=test,
    target_col='price_actual',
    optimize=True,
    sequential=True,  # 逐次予測
    output_path=SUBMISSION,
    filename=filename_sequential
)

[I 2025-06-29 16:16:17,234] A new study created in memory with name: no-name-50bb1049-8f21-4195-a625-62b95eff998c


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 666us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 867us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 636us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 746us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 733us/step


[I 2025-06-29 16:18:38,151] Trial 0 finished with value: 5.074263028514065 and parameters: {'layer1_units': 231, 'layer2_units': 119, 'layer3_units': 41, 'dropout_rate': 0.45816053802580203, 'learning_rate': 0.0017795345905176271, 'batch_size': 16}. Best is trial 0 with value: 5.074263028514065.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 572us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 717us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 855us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 610us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 672us/step


[I 2025-06-29 16:21:10,988] Trial 1 finished with value: 4.793218494337585 and parameters: {'layer1_units': 72, 'layer2_units': 85, 'layer3_units': 8, 'dropout_rate': 0.3424013268551534, 'learning_rate': 0.007644347243250683, 'batch_size': 16}. Best is trial 1 with value: 4.793218494337585.


...

137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 851us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 642us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 688us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 730us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 635us/step


[I 2025-06-29 19:09:02,502] Trial 98 finished with value: 4.898368358090074 and parameters: {'layer1_units': 175, 'layer2_units': 39, 'layer3_units': 15, 'dropout_rate': 0.4468446909021021, 'learning_rate': 0.002900537960298865, 'batch_size': 16}. Best is trial 52 with value: 3.7263147637636926.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 715us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 659us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 609us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 669us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 644us/step


[I 2025-06-29 19:09:50,689] Trial 99 finished with value: 5.024153471111929 and parameters: {'layer1_units': 166, 'layer2_units': 36, 'layer3_units': 24, 'dropout_rate': 0.2663935755146651, 'learning_rate': 0.0033109105067564946, 'batch_size': 64}. Best is trial 52 with value: 3.7263147637636926.


NEURAL_NETWORK：ベイズ最適化の結果
Best params: {'layer1_units': 166, 'layer2_units': 28, 'layer3_units': 21, 'dropout_rate': 0.18465732545534394, 'learning_rate': 0.005130699341029796, 'batch_size': 16, 'hidden_layers': [166, 28, 21], 'epochs': 100, 'patience': 10}
Best CV RMSE: 3.7263147637636926
Epoch 1/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 3s 705us/step - loss: 344.9158 - mae: 13.4244 - learning_rate: 0.0051
Epoch 2/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 657us/step - loss: 138.5943 - mae: 9.2299 - learning_rate: 0.0051
Epoch 3/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 645us/step - loss: 108.5467 - mae: 8.1162 - learning_rate: 0.0051
Epoch 4/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 661us/step - loss: 85.6584 - mae: 7.1422 - learning_rate: 0.0051
Epoch 5/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 647us/step - loss: 73.7934 - mae: 6.6012 - learning_rate: 0.0051
Epoch 6/100
1643/1643 ━━━━━━━━━━━━━━━━━━━━ 1s 729us/step - loss: 63.3315 - mae: 6.1246 - learning_rate: 0.0051
Epoch 7/100
1643/1643 ━━━━━━━━━━━━━━━━━━

### 履歴
___
LIGHTGBM：ベイズ最適化の結果
Best params: {'num_leaves': 21, 'learning_rate': 0.0926973092334304, 'feature_fraction': 0.9533804854356259, 'bagging_fraction': 0.9723501183105292, 'bagging_freq': 10, 'min_child_samples': 43, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1, 'random_state': 42}  
Best CV RMSE: 2.51734282613965  
Selected features: 73/92
___

In [ ]:
predictions_sequential

array([36.01848 , 36.382614, 37.331905, ..., 65.88715 , 63.860985,
       62.69906 ], dtype=float32)

In [16]:
print("finish")

finish
